In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
# Copyright 2017 The TensorFlow Authors All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# =============================================================================


import tensorflow as tf
from tqdm import tqdm

from migration.models import vrnn
from pydantic import PositiveInt
from migration.config import TrainingConfig, DatasetConfig
import migration.datasets as datasets

from migration.models.vrnn_elbo import VRNN
from pathlib import Path

# get batch and model
def create_dataset(cfg: DatasetConfig, repeat : bool) -> tf.data.Dataset:

    return datasets.get_Tensorflow_AIS_dataset(
                    cfg.training_pickle,
                    cfg.batch_size,
                    cfg.encoding_bins.lat,
                    cfg.encoding_bins.lon, 
                    cfg.encoding_bins.sog,
                    cfg.encoding_bins.cog, 
                    shuffle=cfg.shuffle,
                    repeat=repeat)

def create_model(mean_path : Path, latent_size : PositiveInt, total_bins : PositiveInt):
    # Convert the mean of the training set to logit space so it can be used to
    # initialize the bias of the generative distribution.
    mean = datasets.get_AIS_dataset_mean(mean_path)
    generative_bias_init = -tf.math.log(1. / tf.clip_by_value(mean, 0.0001, 0.9999) - 1)
    generative_distribution_class = vrnn.ConditionalBernoulliDistribution
    model = VRNN(total_bins,
                             latent_size,
                             generative_distribution_class,
                             generative_bias_init=generative_bias_init,
                             raw_sigma_bias=0.5, num_samples=1)
    return model



def run_train(cfg : TrainingConfig):

    if cfg.random_seed:
        tf.random.set_seed(cfg.random_seed)

    dataset : tf.data.Dataset = create_dataset(cfg.dataset, repeat=True)    
    model = create_model(cfg.dataset.mean_pickle, cfg.model.latent_size, cfg.dataset.encoding_bins.total)
    return model
    optimizer = tf.keras.optimizers.Adam(learning_rate=cfg.learning_rate)
    
    # @tf.function
    def train_step(x,y, lengths):
        with tf.GradientTape() as tape:
            bound = model((x, y),lengths)
            # Compute lower bounds on the log likelihood.
            bound = tf.reduce_mean(input_tensor=bound / tf.cast(lengths, dtype=tf.float32))
            loss = -bound
        grads = tape.gradient(loss, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        return loss
    
    ckpt = './here.weights.h5'
    for _ in tqdm(range(cfg.epochs)):
        for idx, (inputs, targets, lengths) in enumerate(dataset):
            if Path(ckpt).exists() and idx > 0:
                model.load_weights(ckpt)
            loss = train_step(inputs, targets, lengths)
            model.save_weights(ckpt, overwrite=True)
            if idx % 50 == 0:
                print(loss)


2025-07-10 12:23:57.438053: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752150237.454553 3297520 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752150237.459542 3297520 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1752150237.473908 3297520 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1752150237.473924 3297520 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1752150237.473926 3297520 computation_placer.cc:177] computation placer alr

In [5]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.003)

In [18]:
optimizer.from_config(optimizer.get_config())

In [9]:
epoch = tf.Variable(1)

np.int32(1)

In [22]:
a = epoch.numpy()

In [24]:
epoch.assign_add(2)

<tf.Variable 'UnreadVariable' shape=() dtype=int32, numpy=3>

In [25]:
a

np.int32(1)

In [39]:
epoch.assign(33)

<tf.Variable 'UnreadVariable' shape=() dtype=int32, numpy=33>

In [42]:
epoch.assign(tf.constant(233))

<tf.Variable 'UnreadVariable' shape=() dtype=int32, numpy=233>

3

In [ ]:
print(f'aaa {epoch.numpy()}')

aaa 233


: 

In [33]:
int(epoch) + 1

4

In [31]:
if :
    print(2)

2


In [21]:
for a in range(epoch.numpy(),10):
    print(a)

1
2
3
4
5
6
7
8
9


In [8]:
optimizer.set_weights()

TypeError: BaseOptimizer.set_weights() missing 1 required positional argument: 'weights'

In [ ]:
optimizer.

TypeError: BaseOptimizer.save_own_variables() missing 1 required positional argument: 'store'

In [2]:
d = create_dataset(DatasetConfig(
        training_pickle='../../data/ct_2017010203_10_20/ct_2017010203_10_20_train.pkl',
        mean_pickle='../../data/ct_2017010203_10_20/mean.pkl'
    ), repeat=False)

Instructions for updating:
Use output_signature instead


I0000 00:00:1752141556.622707 3266439 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 40457 MB memory:  -> device: 0, name: NVIDIA L40S-48Q, pci bus id: 0000:00:05.0, compute capability: 8.9


In [6]:
d.save('./here')

In [7]:
datas = tf.data.Dataset.load('./here')

In [8]:
len(datas)

245

In [17]:
from migration.config import TrainingConfig, DatasetConfig

cfg = TrainingConfig(
    dataset=DatasetConfig(
        training_pickle='../../data/ct_2017010203_10_20/ct_2017010203_10_20_train.pkl',
        mean_pickle='../../data/ct_2017010203_10_20/mean.pkl'
    ))

In [9]:
mean_path='../../data/ct_2017010203_10_20/mean.pkl'
import pickle
with open(mean_path,"rb") as f:
    mean = pickle.load(f)

In [12]:
import numpy as np
np.save('./mean.npy', mean)

In [13]:
np.load('./mean.npy')

array([2.38992313e-04, 1.62514773e-04, 2.18677966e-04, 3.01130314e-04,
       2.70061313e-04, 2.84400852e-04, 3.35784200e-04, 2.96350468e-04,
       3.72828008e-04, 3.42953969e-04, 3.82387700e-04, 3.82387700e-04,
       4.25406317e-04, 4.18236547e-04, 4.38550894e-04, 4.79179587e-04,
       4.60060202e-04, 4.91129203e-04, 5.11443549e-04, 5.30562934e-04,
       5.12638511e-04, 5.84336205e-04, 5.81946282e-04, 6.07040475e-04,
       6.23769936e-04, 6.50059091e-04, 6.90687784e-04, 7.01442438e-04,
       7.48045939e-04, 7.45656016e-04, 7.68360286e-04, 7.80309901e-04,
       8.61567288e-04, 8.49617672e-04, 8.65152172e-04, 9.49994443e-04,
       8.97416135e-04, 9.97792906e-04, 9.82258406e-04, 1.08860998e-03,
       1.07427045e-03, 1.11370418e-03, 1.11250922e-03, 1.23917514e-03,
       1.15433287e-03, 1.25351468e-03, 1.25709957e-03, 1.29175345e-03,
       1.31206780e-03, 1.32043253e-03, 1.38376549e-03, 1.32999222e-03,
       1.41124961e-03, 1.39093526e-03, 1.42797907e-03, 1.42439418e-03,
      

In [10]:
import numpy as np


array([2.38992313e-04, 1.62514773e-04, 2.18677966e-04, 3.01130314e-04,
       2.70061313e-04, 2.84400852e-04, 3.35784200e-04, 2.96350468e-04,
       3.72828008e-04, 3.42953969e-04, 3.82387700e-04, 3.82387700e-04,
       4.25406317e-04, 4.18236547e-04, 4.38550894e-04, 4.79179587e-04,
       4.60060202e-04, 4.91129203e-04, 5.11443549e-04, 5.30562934e-04,
       5.12638511e-04, 5.84336205e-04, 5.81946282e-04, 6.07040475e-04,
       6.23769936e-04, 6.50059091e-04, 6.90687784e-04, 7.01442438e-04,
       7.48045939e-04, 7.45656016e-04, 7.68360286e-04, 7.80309901e-04,
       8.61567288e-04, 8.49617672e-04, 8.65152172e-04, 9.49994443e-04,
       8.97416135e-04, 9.97792906e-04, 9.82258406e-04, 1.08860998e-03,
       1.07427045e-03, 1.11370418e-03, 1.11250922e-03, 1.23917514e-03,
       1.15433287e-03, 1.25351468e-03, 1.25709957e-03, 1.29175345e-03,
       1.31206780e-03, 1.32043253e-03, 1.38376549e-03, 1.32999222e-03,
       1.41124961e-03, 1.39093526e-03, 1.42797907e-03, 1.42439418e-03,
      

In [2]:
from migration.config import TrainingConfig, DatasetConfig

cfg = TrainingConfig(
    dataset=DatasetConfig(
        training_pickle='../../data/ct_2017010203_10_20/ct_2017010203_10_20_train.pkl',
        mean_pickle='../../data/ct_2017010203_10_20/mean.pkl'
    ))
# fh = logging.FileHandler(os.path.join(config.logdir,config.log_filename+".log"))
# # get TF logger
# logger = logging.getLogger('tensorflow')
# logger.addHandler(fh)
model = run_train(cfg)


I0000 00:00:1752150244.996756 3297520 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 41768 MB memory:  -> device: 0, name: NVIDIA L40S-48Q, pci bus id: 0000:00:05.0, compute capability: 8.9


In [3]:
model.get_weights()

[]

In [4]:
optimizer.get_weights()

NameError: name 'optimizer' is not defined